In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StandardScaler, Imputer
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
spark = SparkSession.builder \
    .appName("HousePriceClustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [ ]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./big data.csv")

In [ ]:
print(f"✓ Data loaded: {df.count()} rows, {len(df.columns)} columns")
print("\nFirst 5 rows:")
df.select("State", "City", "Price", "Bedroom", "Bathroom", "Area").show(5)



In [ ]:
from pyspark.sql.functions import col

initial_count = df.count()
df = df.filter(col("State").isNotNull() & col("Price").isNotNull())
print(f"✓ Removed {initial_count - df.count()} null rows")

In [ ]:
from pyspark.sql.functions import mean, col, when

print("Step 4: Handling Missing Values...")

# Calculate mean values for numeric columns
print("Calculating mean values...")

# Check which columns exist and are numeric
available_cols = []
for col_name in ["Bedroom", "Bathroom", "Area", "Price", "PPSq"]:
    if col_name in df.columns:
        # Count nulls before
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            print(f"  {col_name}: {null_count} null values")
            available_cols.append(col_name)

# Calculate means only for existing columns
means = {}
for col_name in available_cols:
    mean_val = df.select(mean(col(col_name))).collect()[0][0]
    means[col_name] = mean_val
    print(f"  Mean {col_name}: {mean_val:.2f}")

# Fill missing values manually
for col_name, mean_val in means.items():
    df = df.withColumn(f"{col_name}_imp", 
                      when(col(col_name).isNull(), mean_val).otherwise(col(col_name)))

print(f"✓ Filled missing values for {len(available_cols)} columns")
print(f"  New columns created: {[f'{col}_imp' for col in available_cols]}")

In [ ]:
from pyspark.sql.functions import col

print("Step 5: Removing Price Outliers...")

# Use the price column (after imputation if available)
price_col = "Price_imp" if "Price_imp" in df.columns else "Price"

# Calculate price range
price_stats = df.select(
    mean(col(price_col)).alias("avg_price"),
    stddev(col(price_col)).alias("std_price")
).collect()[0]

avg_price = price_stats["avg_price"]
std_price = price_stats["std_price"]

# Remove prices too far from average
df = df.filter(
    (col(price_col) > avg_price - 3*std_price) & 
    (col(price_col) < avg_price + 3*std_price)
)

print(f"✓ Removed extreme price outliers")
print(f"  Average price: ${avg_price:,.0f}")
print(f"  Standard deviation: ${std_price:,.0f}")

In [ ]:
print("Step 6: Preparing Features for Clustering...")

# Just use the 4 main features
features_to_use = ["Bedroom_imp", "Bathroom_imp", "Area_imp", "Price_imp"]

# Make sure all these columns exist
missing_cols = [col for col in features_to_use if col not in df.columns]
if missing_cols:
    print(f"  Warning: Missing columns {missing_cols}")
    # Use only columns that exist
    features_to_use = [col for col in features_to_use if col in df.columns]

print(f"  Using features: {features_to_use}")

# Import VectorAssembler
from pyspark.ml.feature import VectorAssembler

# Create and apply assembler
assembler = VectorAssembler(inputCols=features_to_use, outputCol="features")
df = assembler.transform(df)

print("✓ Features prepared")
print(f"  Created 'features' column with {len(features_to_use)} features")